In [ ]:
# ============================================================
# CONFIG — edit only this cell
# ============================================================
from pathlib import Path
from mdatools.config import AnalysisConfig, HBondConfig

cfg = AnalysisConfig(
    ligand_resname  = "UNK",
    topology_glob   = "equilibrating_topology.pdb",
    trajectory_glob = "trajectory.xtc",
    dt_ns           = 2.0,
    figures_dir     = Path("./figures"),
    output_dir      = Path("./hbond_results"),
    hbond           = HBondConfig(d_a_cutoff=3.5, angle_cutoff=150.0),
)

REPLICA_ROOTS = [
    # Path("../run01"),
]
# ============================================================

In [ ]:
import pickle
import matplotlib.pyplot as plt
from mdatools.analysis.hbonds import run_hbond_batch

cfg.make_dirs()

results = run_hbond_batch(REPLICA_ROOTS, cfg)

with open(cfg.output_dir / "result_dfs_hbonds.pkl", "wb") as f:
    pickle.dump(results, f)

print(f"Analysed {len(results)} replicas")

In [ ]:
from mdatools.plotting.hbond_plots import (
    plot_occupancy,
    plot_timeseries,
    plot_distance_distribution,
    plot_hbond_count_per_frame,
)

for name, result in results.items():
    if result.summary.empty:
        print(f"{name}: no H-bonds detected")
        continue

    print(f"\n=== {name} ===\n")
    print(result.summary[["hbond_id", "occupancy_%", "mean_dist", "mean_angle"]].head(10).to_string(index=False))

    plot_occupancy(result.summary, name,
                   save_path=cfg.figures_dir / f"hbond_occupancy_{name}.png")
    plt.show()

    plot_timeseries(result.events, result.summary, name,
                    top_n=min(5, len(result.summary)),
                    save_path=cfg.figures_dir / f"hbond_timeseries_{name}.png")
    plt.show()

    plot_distance_distribution(result.events, result.summary, name,
                               top_n=min(5, len(result.summary)),
                               save_path=cfg.figures_dir / f"hbond_distrib_{name}.png")
    plt.show()

    plot_hbond_count_per_frame(result.events, result.n_frames, name,
                               save_path=cfg.figures_dir / f"hbond_count_{name}.png")
    plt.show()